<a href="https://colab.research.google.com/github/gnanendramunagapaka/VoxMind-Intelligent-Voice-AI-Agent/blob/main/VoxMind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install -q faster-whisper
!pip install -q scipy sounddevice
!pip install -q piper-tts
!pip install -q transformers accelerate sentencepiece
!pip install -q IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.9 MB/s eta 0:00:00


In [18]:
import os
import json
import subprocess
import numpy as np

from scipy.io.wavfile import write, read
from IPython.display import Audio, display

print("Libraries imported successfully!")

Libraries imported successfully!


In [20]:
from IPython.display import Javascript, display
from google.colab import output
import base64
import io
import wave

def record_audio_colab(filename="audio.wav", seconds=5):
    print(f"🎤 Recording for {seconds} seconds...")

    js = Javascript(f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
        const recorder = new MediaRecorder(stream);

        const chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);

        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {seconds * 1000}));

        recorder.stop();

        await new Promise(resolve => recorder.onstop = resolve);

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {{type: 'audio/wav'}});
        const buffer = await blob.arrayBuffer();

        const bytes = new Uint8Array(buffer);

        let binary = '';
        bytes.forEach(byte => binary += String.fromCharCode(byte));

        return btoa(binary);
    }}

    recordAudio()
    """)

    data = output.eval_js(js.data)

    audio_bytes = base64.b64decode(data)

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    print("✅ Recording complete!")

    return filename

In [27]:
audio_file = record_audio_colab("audio.wav", seconds=5)

display(Audio("audio.wav"))

🎤 Recording for 5 seconds...
✅ Recording complete!


In [29]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

print("Whisper model loaded successfully!")

Whisper model loaded successfully!


In [31]:
def transcribe_audio(filename="audio.wav"):
    segments, info = whisper_model.transcribe(
        filename,
        beam_size=5
    )

    text = " ".join(
        segment.text for segment in segments
    )

    return text.strip()

In [33]:
segments, info = whisper_model.transcribe(
    "audio.wav",
    language="en"
)

for segment in segments:
    print(segment.text)

 Hi, hello, how are you?


In [35]:
from datetime import datetime
import ast
import operator

def get_current_time():
    return datetime.now().strftime("%I:%M %p")


def calculate(expression):
    try:
        allowed_operators = {
            ast.Add: operator.add,
            ast.Sub: operator.sub,
            ast.Mult: operator.mul,
            ast.Div: operator.truediv,
            ast.Pow: operator.pow,
            ast.Mod: operator.mod,
        }

        def evaluate(node):
            if isinstance(node, ast.Constant):
                return node.value

            if isinstance(node, ast.BinOp):
                left = evaluate(node.left)
                right = evaluate(node.right)

                operation = allowed_operators[type(node.op)]

                return operation(left, right)

            raise ValueError("Unsupported expression")

        tree = ast.parse(expression, mode="eval")

        return str(evaluate(tree.body))

    except Exception:
        return "Unable to calculate the expression."

In [36]:
print("Current time:", get_current_time())

print("Calculation:", calculate("25 * 4 + 10"))

Current time: 04:03 PM
Calculation: 110


In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!


In [39]:
def ask_llm(user_text):

    prompt = f"""
You are a helpful Voice AI Agent.

You have access to these tools:

1. get_current_time
   Use this when the user asks for the current time.

2. calculate
   Use this when the user asks for a mathematical calculation.

If a tool is required, respond ONLY in this JSON format:

{{
    "tool": "tool_name",
    "argument": "tool_argument"
}}

If no tool is required, answer the user normally.

User:
{user_text}

Assistant:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.2,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [40]:
response = ask_llm("What is artificial intelligence?")

print(response)

Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. The term often also has an industrial context, where AI means the use of certain special techniques to enable computers to perform tasks that traditionally require human intelligence, such as visual perception, speech recognition, decision-making, and translation between languages. More specifically, it can be classified into Narrow AI, which is designed to perform specific tasks like image classification or natural language processing; General AI, which would be able to understand and solve any problem; and Superintelligent AI, which could outperform humans


In [41]:
def process_response(response):

    try:
        tool_call = json.loads(response)

        tool_name = tool_call.get("tool")
        argument = tool_call.get("argument", "")

        if tool_name == "get_current_time":

            result = get_current_time()

            return f"The current time is {result}."

        elif tool_name == "calculate":

            result = calculate(argument)

            return f"The result is {result}."

        else:

            return response

    except json.JSONDecodeError:

        return response

In [42]:
user_text = "What is 25 multiplied by 8?"

llm_response = ask_llm(user_text)

print("LLM:")
print(llm_response)

final_response = process_response(llm_response)

print("\nAgent:")
print(final_response)

LLM:
To solve this problem, we need to multiply 25 by 8.
{
    "tool": "calculate",
    "argument": "25 * 8"
}

Agent:
To solve this problem, we need to multiply 25 by 8.
{
    "tool": "calculate",
    "argument": "25 * 8"
}


In [43]:
!pip install -q piper-tts

In [44]:
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json

In [45]:
def speak(text):

    command = [
        "piper",
        "--model",
        "en_US-lessac-medium.onnx",
        "--output_file",
        "response.wav"
    ]

    process = subprocess.Popen(
        command,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    process.communicate(text)

    return "response.wav"

In [46]:
response_file = speak(
    "Hello! I am your voice AI assistant."
)

display(Audio(response_file))

In [47]:
def run_voice_agent():

    print("=" * 50)
    print("       VOICE AI AGENT")
    print("=" * 50)

    print("Say 'exit' or 'quit' to stop.\n")

    while True:

        # 1. Record
        record_audio_colab(
            "audio.wav",
            seconds=5
        )

        # 2. Speech-to-text
        user_text = transcribe_audio(
            "audio.wav"
        )

        print("\nYou:", user_text)

        # 3. Exit
        if user_text.lower().strip() in [
            "exit",
            "quit",
            "stop"
        ]:

            print("Voice Agent stopped.")
            break

        if not user_text:
            print("I didn't hear anything.")
            continue

        # 4. LLM reasoning
        llm_response = ask_llm(
            user_text
        )

        # 5. Tool execution
        final_response = process_response(
            llm_response
        )

        print("Agent:", final_response)

        # 6. Text-to-speech
        response_file = speak(
            final_response
        )

        # 7. Play response
        display(
            Audio(response_file)
        )

In [48]:
run_voice_agent()

       VOICE AI AGENT
Say 'exit' or 'quit' to stop.

🎤 Recording for 5 seconds...
✅ Recording complete!

You: hi whats up
Agent: Hello! How can I assist you today? If you need any information or help with something specific, feel free to ask.


🎤 Recording for 5 seconds...
✅ Recording complete!

You: .  .  .  .


KeyboardInterrupt: 